In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.pipeline import Pipeline
from tqdm import tqdm
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import os
os.chdir('E:/Machine_Learning/Cases/Concrete_Strength/')

In [2]:
df = pd.read_csv('Concrete_Data.csv')
df.head()

,Cement,Blast,Fly,Water,Superplasticizer,Coarse,Fine,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [3]:
X,y = df.drop('Strength',axis=1),df['Strength']
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=26,test_size=0.3)

In [4]:
lr = LinearRegression()
ridge = Ridge()
lasso = Lasso()
dtr = DecisionTreeRegressor(random_state=26)
voting = VotingRegressor([("LR",lr),('Ridge',ridge),('Lasso',lasso),("Tree",dtr)])
voting.fit(X_train,y_train)
y_pred = voting.predict(X_test)
r2_score(y_test,y_pred)

0.7557599363240081

In [5]:
y_pred = voting.named_estimators_['Tree'].predict(X_test)
print('Tree Accuracy : ',r2_score(y_test,y_pred))

y_pred = voting.named_estimators_['LR'].predict(X_test)
print('LR Accuracy : ',r2_score(y_test,y_pred))

y_pred = voting.named_estimators_['Ridge'].predict(X_test)
print('Ridge Accuracy : ',r2_score(y_test,y_pred))

y_pred = voting.named_estimators_['Lasso'].predict(X_test)
print('Lasso Accuracy : ',r2_score(y_test,y_pred))

Tree Accuracy :  0.8500256114787615
LR Accuracy :  0.6181464728464399
Ridge Accuracy :  0.6181470326108837
Lasso Accuracy :  0.6188405404462809


In [6]:
treewt = np.linspace(1,5,10)
lrwt = np.linspace(1,5,10)
ridgewt = np.linspace(1,5,10)
lassowt = np.linspace(1,5,10)

In [7]:
from tqdm import tqdm

In [9]:
scores = []
for t in tqdm(treewt):
    for l in lrwt:
        for r in ridgewt:
            for ls in lassowt:
                voting = VotingRegressor([("Tree",dtr),("LR",lr),('Ridge',ridge),('Lasso',lasso)],weights=[t,l,r,ls])
                voting.fit(X_train,y_train)
                y_pred = voting.predict(X_test)
                scores.append([t,l,r,ls,r2_score(y_test,y_pred)])


df_s = pd.DataFrame(scores,columns=['weightTree','weightLR','weightRidge','weightLasso','r2'])
df_s.sort_values('r2',ascending=False)

100%|██████████| 10/10 [02:46<00:00, 16.65s/it]


,weightTree,weightLR,weightRidge,weightLasso,r2
9000,5.000000,1.000000,1.000000,1.000000,0.862472
8000,4.555556,1.000000,1.000000,1.000000,0.859494
9001,5.000000,1.000000,1.000000,1.444444,0.857896
9010,5.000000,1.000000,1.444444,1.000000,0.857871
9100,5.000000,1.444444,1.000000,1.000000,0.857871
...,...,...,...,...,...
997,1.000000,5.000000,5.000000,4.111111,0.659899
899,1.000000,4.555556,5.000000,5.000000,0.658800
989,1.000000,5.000000,4.555556,5.000000,0.658800
998,1.000000,5.000000,5.000000,4.555556,0.658780
